# Características RoCoLe — inventario con máscara COCO

Objetivo: armar un inventario de las mismas 9 medidas de color y textura que el cuaderno 02, pero midiendo solo dentro de la máscara oficial Leaf de COCO, y guardar un CSV paralelo. No hay EDA largo ni modelos: es un inventario masked autocontenido.

Responsables: Víctor y Gio.


## Llegar al índice, al COCO y fijar la semilla

Usamos la carpeta featured, el índice partido, el JSON oficial de anotaciones y una carpeta de figuras.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from pycocotools.coco import COCO
from skimage.feature import graycomatrix, graycoprops
from tqdm.auto import tqdm


In [ ]:
# semilla fija para muestreos reproducibles
seed = 42
repo_root = Path("../..").resolve()
featured_dir = repo_root / "data" / "featured" / "rocole"
index_path = featured_dir / "index.csv"
coco_json_path = (
    repo_root / "data" / "raw" / "RoCoLe" / "Annotations" / "RoCoLE-coco.json"
)
results_dir = repo_root / "experiments" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

print("repo_root:", repo_root)
print("index_path:", index_path)
print("existe índice:", index_path.exists())
print("coco_json_path:", coco_json_path)
print("existe coco:", coco_json_path.exists())
print("results_dir:", results_dir)
seed


## Cargar el índice partido

El índice es la mesa de trabajo: una fila por foto, con etiqueta, partición y ruta. No rehacemos el split.


In [ ]:
rocole_index = pd.read_csv(index_path)

print("filas:", len(rocole_index))
print("columnas:", list(rocole_index.columns))
rocole_index.head()


## Columnas de identidad que viajan con cada fila

Además de las medidas, cada fila del inventario lleva archivo, split, grupo de enlace y las dos etiquetas del índice.


In [ ]:
# rocole_index: mesa con File, split, link_group y etiquetas

print("filas:", len(rocole_index))
print("splits:", rocole_index["split"].value_counts().to_dict())
id_columns = [
    "File",
    "split",
    "link_group",
    "Binary.Label",
    "Multiclass.Label",
]

print("id_columns:", id_columns)
rocole_index[id_columns].head()


## Cargar las anotaciones COCO

Usamos el JSON oficial de segmentación para saber qué píxeles son hoja en cada foto.


In [ ]:
# si falta: pip install pycocotools

coco = COCO(str(coco_json_path))

print("coco_json_path:", coco_json_path)
print("imágenes en COCO:", len(coco.getImgIds()))
coco


## Constantes de medición

Fijamos el eps de los índices, los 32 niveles de gris del GLCM y la lista de las nueve columnas de features que debe devolver el inventario.


In [ ]:
eps = 1e-8
gray_level_count = 32
feature_columns = [
    "exg_mean",
    "exg_std",
    "ngrdi_mean",
    "ngrdi_std",
    "vari_mean",
    "vari_std",
    "glcm_contrast",
    "glcm_homogeneity",
    "glcm_energy",
]

print("eps:", eps)
print("gray_level_count:", gray_level_count)
feature_columns


## Cargar la máscara COCO por File

En el JSON, file_name es una URL; emparejamos si termina en el File del índice. Devolvemos la unión (OR) de las anotaciones Leaf.


In [ ]:
# coco: anotaciones RoCoLe ya cargadas

def load_coco_leaf_mask(coco, file_name, height, width):
    """Máscara Leaf (OR) emparejando file_name del JSON por sufijo File."""
    matched_images = [
        img
        for img in coco.loadImgs(coco.getImgIds())
        if img["file_name"].endswith(file_name)
    ]
    if len(matched_images) != 1:
        raise ValueError(
            f"esperaba 1 imagen COCO para {file_name}, hay {len(matched_images)}"
        )

    coco_image = matched_images[0]
    # categoría Leaf = 1; imgIds en lista (ids Labelbox son strings)
    ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=[1])
    if len(ann_ids) < 1:
        # respaldo por nombre de categoría si catIds=[1] no encuentra
        leaf_cat_ids = [
            cat["id"]
            for cat in coco.loadCats(coco.getCatIds())
            if cat.get("name", "").lower() == "leaf"
        ]
        ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=leaf_cat_ids)

    annotations = coco.loadAnns(ann_ids)
    if len(annotations) < 1:
        raise ValueError(f"sin anotaciones Leaf para {file_name}")

    leaf_mask = None
    for annotation in annotations:
        mask_i = coco.annToMask(annotation).astype(bool)
        leaf_mask = mask_i if leaf_mask is None else (leaf_mask | mask_i)

    if leaf_mask.shape != (height, width):
        raise ValueError(
            f"máscara {leaf_mask.shape} vs imagen {(height, width)} para {file_name}"
        )
    return leaf_mask


load_coco_leaf_mask


## Resumir color solo dentro de la hoja

Media y desviación de ExG, NGRDI y VARI únicamente en los píxeles de la máscara. Si la máscara está vacía, no hay números válidos: error explícito.


In [ ]:
# eps ya fijado arriba

def extract_rgb_summary(image, leaf_mask):
    """Media y std de ExG, NGRDI y VARI solo en píxeles de la máscara."""
    if hasattr(image, "convert"):
        image_rgb = np.asarray(image.convert("RGB"), dtype=np.float64)
    else:
        image_rgb = np.asarray(image, dtype=np.float64)

    leaf_mask = np.asarray(leaf_mask, dtype=bool)
    if leaf_mask.sum() == 0:
        raise ValueError("máscara vacía: no hay píxeles de hoja para RGB")

    red_channel = image_rgb[:, :, 0]
    green_channel = image_rgb[:, :, 1]
    blue_channel = image_rgb[:, :, 2]

    channel_sum = red_channel + green_channel + blue_channel + eps
    r = red_channel / channel_sum
    g = green_channel / channel_sum
    b = blue_channel / channel_sum
    exg_map = 2.0 * g - r - b
    ngrdi_map = (green_channel - red_channel) / (green_channel + red_channel + eps)
    vari_map = (green_channel - red_channel) / (
        green_channel + red_channel - blue_channel + eps
    )

    return {
        "exg_mean": float(exg_map[leaf_mask].mean()),
        "exg_std": float(exg_map[leaf_mask].std()),
        "ngrdi_mean": float(ngrdi_map[leaf_mask].mean()),
        "ngrdi_std": float(ngrdi_map[leaf_mask].std()),
        "vari_mean": float(vari_map[leaf_mask].mean()),
        "vari_std": float(vari_map[leaf_mask].std()),
    }


extract_rgb_summary


## Resumir textura GLCM solo en la hoja

Recortamos al bounding box de la máscara, ponemos en 0 lo que no es hoja y calculamos contraste, homogeneidad y energía con distancia 1, ángulo 0 y 32 niveles.


In [ ]:
# gray_level_count ya fijado arriba

def extract_glcm_summary(image, leaf_mask):
    """GLCM d=1, ángulo 0, 32 niveles sobre bbox con fuera-hoja en 0."""
    if hasattr(image, "convert"):
        gray = np.asarray(image.convert("L"), dtype=np.uint8)
    else:
        gray = np.asarray(image, dtype=np.uint8)

    leaf_mask = np.asarray(leaf_mask, dtype=bool)
    if leaf_mask.sum() == 0:
        raise ValueError("máscara vacía: no hay píxeles de hoja para GLCM")

    rows, cols = np.where(leaf_mask)
    y0, y1 = int(rows.min()), int(rows.max()) + 1
    x0, x1 = int(cols.min()), int(cols.max()) + 1
    work_gray = gray[y0:y1, x0:x1].copy()
    crop_mask = leaf_mask[y0:y1, x0:x1]
    # fuera de hoja dentro del bbox → 0 (constante)
    work_gray[~crop_mask] = 0

    quantized_gray = (
        work_gray.astype(np.float64) / 255.0 * (gray_level_count - 1)
    ).astype(np.uint8)

    glcm = graycomatrix(
        quantized_gray,
        distances=[1],
        angles=[0],
        levels=gray_level_count,
    )
    return {
        "glcm_contrast": float(graycoprops(glcm, "contrast")[0, 0]),
        "glcm_homogeneity": float(graycoprops(glcm, "homogeneity")[0, 0]),
        "glcm_energy": float(graycoprops(glcm, "energy")[0, 0]),
    }


extract_glcm_summary


## Filas de smoke (train)

Para probar el camino sin recorrer todo el índice, tomamos seis fotos de aprender: tres sanas y tres afectadas, con la semilla fija.


In [ ]:
# rocole_index, seed, id_columns

train_rows = rocole_index.loc[rocole_index["split"] == "train"].reset_index(drop=True)
healthy_train_sample = (
    train_rows.loc[train_rows["Binary.Label"] == "healthy"]
    .sample(n=3, random_state=seed)
    .reset_index(drop=True)
)
unhealthy_train_sample = (
    train_rows.loc[train_rows["Binary.Label"] == "unhealthy"]
    .sample(n=3, random_state=seed)
    .reset_index(drop=True)
)
smoke_train_rows = pd.concat(
    [healthy_train_sample, unhealthy_train_sample],
    ignore_index=True,
)

print("smoke_train_rows:", len(smoke_train_rows))
print(smoke_train_rows["Binary.Label"].value_counts().to_string())
smoke_train_rows[id_columns]


## Smoke: features con máscara, fila a fila

Sobre esas pocas filas de train leemos la foto, pedimos la máscara Leaf y medimos RGB+GLCM. Si falla (sin máscara u otro error), NaN en features y mensaje en error. No caemos a foto completa.


In [ ]:
# smoke_train_rows, coco, repo_root
# load_coco_leaf_mask, extract_rgb_summary, extract_glcm_summary
# feature_columns, id_columns

smoke_feature_rows = []
for _, row in smoke_train_rows.iterrows():
    feature_row = {column_name: row[column_name] for column_name in id_columns}
    feature_row["error"] = None
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        height, width = sample_image.size[1], sample_image.size[0]
        leaf_mask = load_coco_leaf_mask(coco, row["File"], height, width)
        rgb_summary = extract_rgb_summary(sample_image, leaf_mask)
        glcm_summary = extract_glcm_summary(sample_image, leaf_mask)
        feature_row.update(rgb_summary)
        feature_row.update(glcm_summary)
    except Exception as exc:
        # fallo de máscara o lectura: NaN en features, mensaje en error
        feature_row["error"] = str(exc)

    smoke_feature_rows.append(feature_row)

print("smoke_feature_rows:", len(smoke_feature_rows))
print("con error:", sum(1 for r in smoke_feature_rows if r["error"]))
smoke_feature_rows


## Tabla smoke y chequeo de columnas / finitos

Pasamos el smoke a DataFrame y comprobamos que estén todas las columnas pedidas y que, sin error, los números sean finitos.


In [ ]:
# smoke_feature_rows, feature_columns, id_columns

smoke_feature_table = pd.DataFrame(smoke_feature_rows)

expected_columns = id_columns + feature_columns + ["error"]
missing_columns = [
    name for name in expected_columns if name not in smoke_feature_table.columns
]
assert missing_columns == [], f"faltan columnas: {missing_columns}"

ok_mask = smoke_feature_table["error"].isna()
finite_ok = True
if ok_mask.any():
    feature_block = smoke_feature_table.loc[ok_mask, feature_columns]
    finite_ok = bool(np.isfinite(feature_block.to_numpy(dtype=float)).all())

print("filas:", len(smoke_feature_table))
print("columnas esperadas ok:", missing_columns == [])
print("finitos en filas sin error:", finite_ok)
smoke_feature_table


## Barrido completo del índice (con máscara)

Recorremos todas las filas de rocole_index con el mismo protocolo masked. Si no hay máscara Leaf, queda error y NaN: no caemos a foto completa ni miramos el test para elegir nada.


In [ ]:
# rocole_index, coco, repo_root
# load_coco_leaf_mask, extract_rgb_summary, extract_glcm_summary
# feature_columns, id_columns

try:
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

inventory_feature_rows = []
for _, row in tqdm(
    rocole_index.iterrows(),
    total=len(rocole_index),
    desc="inventario RGB+GLCM masked",
):
    feature_row = {column_name: row[column_name] for column_name in id_columns}
    feature_row["error"] = None
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        height, width = sample_image.size[1], sample_image.size[0]
        # sin fallback a full-image: si no hay máscara, Exception → error
        leaf_mask = load_coco_leaf_mask(coco, row["File"], height, width)
        rgb_summary = extract_rgb_summary(sample_image, leaf_mask)
        glcm_summary = extract_glcm_summary(sample_image, leaf_mask)
        feature_row.update(rgb_summary)
        feature_row.update(glcm_summary)
    except Exception as exc:
        feature_row["error"] = str(exc)

    inventory_feature_rows.append(feature_row)

print("inventory_feature_rows:", len(inventory_feature_rows))
print(
    "con error:",
    sum(1 for r in inventory_feature_rows if r["error"] is not None),
)


## Mesa de features del inventario masked

Convertimos la lista del barrido en un DataFrame: rgb_glcm_features_masked.


In [ ]:
# inventory_feature_rows

rgb_glcm_features_masked = pd.DataFrame(inventory_feature_rows)

print("filas:", len(rgb_glcm_features_masked))
print("columnas:", list(rgb_glcm_features_masked.columns))
rgb_glcm_features_masked.head()


## Chequeo de filas del inventario

Comprobamos que haya una fila por foto del índice, que los conteos por split cuadren y cuántas filas trajeron error.


In [ ]:
# rgb_glcm_features_masked, rocole_index

inventory_check = {
    "n_index": len(rocole_index),
    "n_features": len(rgb_glcm_features_masked),
    "len_match": len(rgb_glcm_features_masked) == len(rocole_index),
    "split_counts_index": rocole_index["split"].value_counts().to_dict(),
    "split_counts_features": rgb_glcm_features_masked["split"]
    .value_counts()
    .to_dict(),
    "n_errors": int(rgb_glcm_features_masked["error"].notna().sum()),
}

assert inventory_check["len_match"], (
    f"filas features {inventory_check['n_features']} "
    f"!= índice {inventory_check['n_index']}"
)

inventory_check


## Guardar el CSV de features masked

Escribimos rgb_glcm_features_masked.csv en la carpeta featured de RoCoLe. No entrenamos nada aquí.


In [ ]:
# rgb_glcm_features_masked, featured_dir

features_csv_path = featured_dir / "rgb_glcm_features_masked.csv"
rgb_glcm_features_masked.to_csv(features_csv_path, index=False)

print("guardado:", features_csv_path)
print("existe:", features_csv_path.exists())
features_csv_path


## Conclusiones

Quedó un CSV paralelo de las mismas nueve medidas RGB+GLCM, pero solo dentro de la máscara oficial Leaf de COCO; si faltaba máscara, la fila quedó con error y NaN, sin caer a foto completa ni entrenar modelos.
